# Module 2F — Caltech Temporal Pedestrian Localization

- **Dataset:** [Caltech Pedestrian YOLO](https://www.kaggle.com/datasets/abhinavsasikumar/caltech-pedestrian-yolo/data)
- **Task:** use consecutive frames to localize the primary pedestrian
- **Model:** YOLOv8n feature backbone + LSTM
- **Output:** normalized `(x_center, y_center, width, height)` and pedestrian confidence
- **Export:** `best_pedestrian_yololstm.pt`

This model does **not** predict crossing intent because Caltech supplies detection boxes, not crossing/not-crossing labels. It complements the normal YOLO + ByteTrack path by adding temporal confirmation and short-occlusion recovery.

In [ ]:
!pip install -q ultralytics opendatasets scikit-learn

import os
import random
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEQUENCE_LENGTH = 5
INPUT_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 1e-4
CONFIDENCE_THRESHOLD = 0.50

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Optional Google Drive checkpoint storage

Mount Drive in Colab so the best checkpoint survives a disconnected session. Skip this cell on Kaggle and set `OUTPUT_DIR` to `/kaggle/working` instead.

In [ ]:
IN_COLAB = Path("/content").exists() and "COLAB_RELEASE_TAG" in os.environ
IN_KAGGLE = Path("/kaggle/working").exists()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/bmw_ai/models")
elif IN_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working")
else:
    OUTPUT_DIR = Path("./outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "best_pedestrian_yololstm.pt"
print("Checkpoint:", OUTPUT_PATH)

## 2. Download the Caltech Pedestrian YOLO dataset

The Kaggle dataset contains frame images and YOLO text labels. Kaggle notebooks can attach it through **Add Input**; Colab can download it with `opendatasets` and Kaggle credentials.

### Dataset source

On Kaggle, attach `abhinavsasikumar/caltech-pedestrian-yolo` and the next cell will discover it under `/kaggle/input`. On Colab, the next cell downloads the same public dataset.

In [ ]:
DATASET_URL = "https://www.kaggle.com/datasets/abhinavsasikumar/caltech-pedestrian-yolo"

if IN_KAGGLE:
    candidates = list(Path("/kaggle/input").glob("*caltech*pedestrian*yolo*"))
    if not candidates:
        raise FileNotFoundError(
            "Attach the Caltech Pedestrian YOLO dataset with Kaggle's Add Input button."
        )
    DATA_ROOT = candidates[0]
else:
    import opendatasets as od
    download_root = Path("/content") if IN_COLAB else Path("./data")
    od.download(DATASET_URL, data_dir=str(download_root))
    candidates = list(download_root.glob("*caltech*pedestrian*yolo*"))
    if not candidates:
        raise FileNotFoundError("Downloaded dataset directory was not found")
    DATA_ROOT = candidates[0]

print("Dataset root:", DATA_ROOT)


## 3. Discover frame/label pairs and temporal sequences

Files are grouped by their set/video prefix and sorted by the trailing frame number. Splitting happens at the sequence-group level to prevent neighboring frames leaking between train and validation sets.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
images = [p for p in DATA_ROOT.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
label_index = defaultdict(list)
for label_path in DATA_ROOT.rglob("*.txt"):
    label_index[label_path.stem].append(label_path)

# A missing/empty YOLO label is a valid negative frame for the confidence head.
pairs = []
for image_path in images:
    matches = label_index.get(image_path.stem, [])
    pairs.append((image_path, matches[0] if matches else None))

if not pairs:
    raise RuntimeError("No frame images were found")


def temporal_identity(path: Path):
    """Return (sequence key, sortable frame number) from common Caltech names."""
    stem = path.stem
    match = re.match(r"^(.*?)(?:[_-](?:I|frame)?)(\d+)$", stem, flags=re.I)
    if match is None:
        match = re.match(r"^(.*?)(\d+)$", stem)
    if match:
        prefix, frame_number = match.group(1), int(match.group(2))
    else:
        prefix, frame_number = stem, 0
    relative_parent = str(path.parent.relative_to(DATA_ROOT))
    return f"{relative_parent}/{prefix}", frame_number


groups = defaultdict(list)
for image_path, label_path in pairs:
    group_key, frame_number = temporal_identity(image_path)
    groups[group_key].append((frame_number, image_path, label_path))

for frames in groups.values():
    frames.sort(key=lambda item: item[0])

groups = {key: value for key, value in groups.items() if len(value) >= SEQUENCE_LENGTH}
if not groups:
    raise RuntimeError(
        "No temporal groups contain enough frames. Inspect filename grouping before training."
    )

print("Frame images:", len(pairs))
print("Positive label files:", sum(label is not None for _, label in pairs))
print("Temporal groups:", len(groups))
print("Example group length:", len(next(iter(groups.values()))))

## 4. Build leakage-safe train/validation/test windows

Each target is the largest annotated pedestrian in the final frame of a sequence. The confidence target is `1` when a pedestrian exists and `0` for an empty frame. This matches the checkpoint's single bbox head and confidence head.

In [ ]:
def read_primary_bbox(label_path: Path | None):
    if label_path is None or not label_path.is_file():
        return np.zeros(4, dtype=np.float32), 0.0
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) < 5:
            continue
        _, x_center, y_center, width, height = map(float, fields[:5])
        box = np.clip([x_center, y_center, width, height], 0.0, 1.0)
        if box[2] > 0 and box[3] > 0:
            boxes.append(box)
    if not boxes:
        return np.zeros(4, dtype=np.float32), 0.0
    primary = max(boxes, key=lambda box: box[2] * box[3])
    return np.asarray(primary, dtype=np.float32), 1.0


def build_windows(group_items):
    windows = []
    for group_key, frames in group_items:
        for end in range(SEQUENCE_LENGTH - 1, len(frames)):
            window = frames[end - SEQUENCE_LENGTH + 1 : end + 1]
            windows.append((group_key, window))
    return windows


group_keys = sorted(groups)
train_keys, holdout_keys = train_test_split(
    group_keys, test_size=0.20, random_state=SEED
)
val_keys, test_keys = train_test_split(
    holdout_keys, test_size=0.50, random_state=SEED
)

train_windows = build_windows([(key, groups[key]) for key in train_keys])
val_windows = build_windows([(key, groups[key]) for key in val_keys])
test_windows = build_windows([(key, groups[key]) for key in test_keys])

print({
    "train_groups": len(train_keys), "train_windows": len(train_windows),
    "val_groups": len(val_keys), "val_windows": len(val_windows),
    "test_groups": len(test_keys), "test_windows": len(test_windows),
})

## 5. Sequence dataset and data loaders

A horizontal flip is applied consistently across every frame in a training window, and the normalized target center is mirrored with it.

In [ ]:
class CaltechTemporalDataset(Dataset):
    def __init__(self, windows, augment=False):
        self.windows = windows
        self.augment = augment

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        _, window = self.windows[index]
        flip = self.augment and random.random() < 0.5
        frames = []
        for _, image_path, _ in window:
            image = Image.open(image_path).convert("RGB")
            image = TF.resize(image, [INPUT_SIZE, INPUT_SIZE])
            tensor = TF.to_tensor(image)
            if flip:
                tensor = torch.flip(tensor, dims=[2])
            frames.append(tensor)

        bbox, confidence = read_primary_bbox(window[-1][2])
        if flip and confidence > 0:
            bbox[0] = 1.0 - bbox[0]
        return (
            torch.stack(frames),
            torch.from_numpy(bbox),
            torch.tensor(confidence, dtype=torch.float32),
        )


train_ds = CaltechTemporalDataset(train_windows, augment=True)
val_ds = CaltechTemporalDataset(val_windows)
test_ds = CaltechTemporalDataset(test_windows)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

sample_frames, sample_bbox, sample_conf = train_ds[0]
print("frames:", sample_frames.shape)
print("target bbox/conf:", sample_bbox.tolist(), sample_conf.item())

## 6. YOLOv8n backbone + LSTM

The first ten YOLOv8n modules produce a `256×7×7` feature map for each 224px frame. The LSTM consumes five flattened feature maps and predicts the primary pedestrian box/confidence for the final frame. This definition matches `ml/models/best_pedestrian_yololstm.pt`.

In [ ]:
class TemporalPedestrianYOLOLSTM(nn.Module):
    def __init__(self, hidden_size=256, pretrained_backbone=True):
        super().__init__()
        source = "yolov8n.pt" if pretrained_backbone else "yolov8n.yaml"
        yolo = YOLO(source)
        self.backbone = nn.Sequential(*list(yolo.model.model.children())[:10])
        self.lstm = nn.LSTM(
            input_size=256 * 7 * 7,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.fc_bbox = nn.Linear(hidden_size, 4)
        self.fc_conf = nn.Linear(hidden_size, 1)

    def forward(self, frames):
        batch, steps, channels, height, width = frames.shape
        features = self.backbone(
            frames.reshape(batch * steps, channels, height, width)
        )
        features = features.flatten(1).reshape(batch, steps, -1)
        _, (hidden, _) = self.lstm(features)
        temporal = hidden[-1]
        bbox_xywh = torch.sigmoid(self.fc_bbox(temporal))
        confidence = torch.sigmoid(self.fc_conf(temporal)).squeeze(-1)
        return bbox_xywh, confidence


model = TemporalPedestrianYOLOLSTM().to(DEVICE)
bbox_loss_fn = nn.SmoothL1Loss(reduction="none")
confidence_loss_fn = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 7. Train and validate

The bbox regression loss is applied only to positive final frames. Confidence BCE handles both positive and empty frames. Validation reports confidence accuracy and mean IoU for positive examples.

In [ ]:
def bbox_iou_xywh(predicted, target):
    pred_x1 = predicted[:, 0] - predicted[:, 2] / 2
    pred_y1 = predicted[:, 1] - predicted[:, 3] / 2
    pred_x2 = predicted[:, 0] + predicted[:, 2] / 2
    pred_y2 = predicted[:, 1] + predicted[:, 3] / 2
    true_x1 = target[:, 0] - target[:, 2] / 2
    true_y1 = target[:, 1] - target[:, 3] / 2
    true_x2 = target[:, 0] + target[:, 2] / 2
    true_y2 = target[:, 1] + target[:, 3] / 2
    intersection = (
        (torch.minimum(pred_x2, true_x2) - torch.maximum(pred_x1, true_x1)).clamp_min(0)
        * (torch.minimum(pred_y2, true_y2) - torch.maximum(pred_y1, true_y1)).clamp_min(0)
    )
    pred_area = predicted[:, 2] * predicted[:, 3]
    true_area = target[:, 2] * target[:, 3]
    return intersection / (pred_area + true_area - intersection).clamp_min(1e-6)


def run_epoch(loader, training=False):
    model.train(training)
    totals = {"loss": 0.0, "correct": 0, "count": 0, "iou_sum": 0.0, "positive": 0}
    for frames, target_bbox, target_conf in tqdm(loader, leave=False):
        frames = frames.to(DEVICE, non_blocking=True)
        target_bbox = target_bbox.to(DEVICE, non_blocking=True)
        target_conf = target_conf.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            predicted_bbox, predicted_conf = model(frames)
            positive = target_conf > 0.5
            bbox_loss = bbox_loss_fn(predicted_bbox, target_bbox).mean(dim=1)
            bbox_loss = bbox_loss[positive].mean() if positive.any() else bbox_loss.sum() * 0
            conf_loss = confidence_loss_fn(predicted_conf, target_conf)
            loss = 5.0 * bbox_loss + conf_loss
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        batch_size = len(frames)
        totals["loss"] += loss.item() * batch_size
        totals["correct"] += ((predicted_conf >= CONFIDENCE_THRESHOLD) == positive).sum().item()
        totals["count"] += batch_size
        if positive.any():
            totals["iou_sum"] += bbox_iou_xywh(
                predicted_bbox[positive], target_bbox[positive]
            ).sum().item()
            totals["positive"] += positive.sum().item()
    return {
        "loss": totals["loss"] / max(totals["count"], 1),
        "confidence_accuracy": totals["correct"] / max(totals["count"], 1),
        "mean_iou": totals["iou_sum"] / max(totals["positive"], 1),
    }


best_score = -1.0
for epoch in range(EPOCHS):
    train_metrics = run_epoch(train_loader, training=True)
    with torch.inference_mode():
        val_metrics = run_epoch(val_loader)
    scheduler.step()
    score = val_metrics["mean_iou"] + val_metrics["confidence_accuracy"]
    print(
        f"Epoch {epoch + 1:02d} | train loss {train_metrics['loss']:.4f} | "
        f"val IoU {val_metrics['mean_iou']:.4f} | "
        f"val conf acc {val_metrics['confidence_accuracy']:.4f}"
    )
    if score > best_score:
        best_score = score
        checkpoint = {
            "model_state_dict": {key: value.cpu() for key, value in model.state_dict().items()},
            "model_class": "TemporalPedestrianYOLOLSTM",
            "model_config": {"input_size": INPUT_SIZE, "hidden_size": 256},
            "sequence_length": SEQUENCE_LENGTH,
            "bbox_format": "normalized_xywh",
            "dataset": DATASET_URL,
            "task": "primary_pedestrian_temporal_localization",
            "epoch": epoch + 1,
            "validation": val_metrics,
        }
        torch.save(checkpoint, OUTPUT_PATH)
        print("  Saved:", OUTPUT_PATH)

print("Best validation score:", best_score)

In [ ]:
saved = torch.load(OUTPUT_PATH, map_location=DEVICE, weights_only=False)
reloaded = TemporalPedestrianYOLOLSTM(pretrained_backbone=False).to(DEVICE)
reloaded.load_state_dict(saved["model_state_dict"], strict=True)
model = reloaded
with torch.inference_mode():
    test_metrics = run_epoch(test_loader)

print("Test metrics:", test_metrics)
print("Task:", saved["task"])
print("Dataset:", saved["dataset"])
print("Sequence length:", saved["sequence_length"])

if IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
else:
    print("Copy this checkpoint into BMW/ml/models:", OUTPUT_PATH)

In [ ]:
## Deployment contract and limitation

Place `best_pedestrian_yololstm.pt` in `ml/models/`. Module 2F buffers five frames, predicts one primary normalized pedestrian box plus confidence, and can associate that box with an existing pedestrian track.

Caltech does not include crossing-intent labels. Consequently this checkpoint must never be presented as `crossing` / `not crossing`, and safety rules must use observed pedestrian proximity and motion rather than inferred intent.